# Sentiment Analysis Steam Reviews - Preprocessing

## 1. Setup

### 1.1 Imports & Settings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
pd.set_option('display.max_colwidth', 200) # see full review text
sns.set_theme(style = 'darkgrid')

### 1.2 Load raw data

In [3]:
data = pd.read_csv('../data/raw_data/steam_reviews.csv')

print(f'Raw data: {data.shape}')

Raw data: (434891, 8)


## 2. Target & Types

### 2.1 Create target (label)

In [4]:
data['label'] = data['recommendation'].map({'Recommended': 1, 'Not Recommended': 0})

data.head()

,date_posted,funny,helpful,hour_played,is_early_access_review,recommendation,review,title,label
0,2019-02-10,2,4,578,False,Recommended,&gt Played as German Reich&gt Declare war on Belgium&gt Can't break Belgium so go through France&gt Capitulate France in order to get to Belgium&gt Get True Blitzkrieg achievementThis game is dad,Expansion - Hearts of Iron IV: Man the Guns,1
1,2019-02-10,0,0,184,False,Recommended,yes.,Expansion - Hearts of Iron IV: Man the Guns,1
2,2019-02-07,0,0,892,False,Recommended,Very good game although a bit overpriced in my opinion. I'd prefer playing the game with mods (historical accuracy so on) although the vanilla version is good aswell. 7/10,Expansion - Hearts of Iron IV: Man the Guns,1
3,2018-06-14,126,1086,676,False,Recommended,Out of all the reviews I wrote This one is probably the most serious one I wrote. For starters the community of this game sucks just like every online game You don't wanna talk to them because the...,Dead by Daylight,1
4,2017-06-20,85,2139,612,False,Recommended,Disclaimer I survivor main. I play games for fun not for competition so the DBD community doesn't really get to me. If I get a bad killer that face camps oh well. I die and move on to the next gam...,Dead by Daylight,1


In [5]:
print(data['label'].value_counts(dropna = False))

label
1    303593
0    131298
Name: count, dtype: int64


### 2.2 Handle funny artifacts

In [6]:
MAX_VALID_FUNNY = 1_000_000
data.loc[data['funny'] > MAX_VALID_FUNNY, 'funny'] = np.nan

print(f'Nulled {data['funny'].isna().sum()} invalid funny values')

Nulled 54 invalid funny values


## 3. Drop rows without review text

In [7]:
before = len(data)

data = data.dropna(subset = ['review']).reset_index(drop = True)

print(f'Dropped {before - len(data)} rows without review text')
print(f'Remaining: {len(data)}')

Dropped 1516 rows without review text
Remaining: 433375


## 4. Text cleaning

### 4.1 Set up the preprocessing module

In [8]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from src.preprocessing import clean_text

In [9]:
# Sample testing

sample = data['review'].iloc[0]
print('RAW:  ', repr(sample[:80]))
print('CLEAN:', repr(clean_text(sample)[:80]))

RAW:   "&gt Played as German Reich&gt Declare war on Belgium&gt Can't break Belgium so g"
CLEAN: "> played as german reich> declare war on belgium> can't break belgium so go thro"


In [10]:
data['review_clean'] = data['review'].apply(clean_text)

In [11]:
data[['review', 'review_clean']].head(10)

,review,review_clean
0,&gt Played as German Reich&gt Declare war on Belgium&gt Can't break Belgium so go through France&gt Capitulate France in order to get to Belgium&gt Get True Blitzkrieg achievementThis game is dad,> played as german reich> declare war on belgium> can't break belgium so go through france> capitulate france in order to get to belgium> get true blitzkrieg achievementthis game is dad
1,yes.,yes.
2,Very good game although a bit overpriced in my opinion. I'd prefer playing the game with mods (historical accuracy so on) although the vanilla version is good aswell. 7/10,very good game although a bit overpriced in my opinion. i'd prefer playing the game with mods (historical accuracy so on) although the vanilla version is good aswell. 7/10
3,Out of all the reviews I wrote This one is probably the most serious one I wrote. For starters the community of this game sucks just like every online game You don't wanna talk to them because the...,out of all the reviews i wrote this one is probably the most serious one i wrote. for starters the community of this game sucks just like every online game you don't wanna talk to them because the...
4,Disclaimer I survivor main. I play games for fun not for competition so the DBD community doesn't really get to me. If I get a bad killer that face camps oh well. I die and move on to the next gam...,disclaimer i survivor main. i play games for fun not for competition so the dbd community doesn't really get to me. if i get a bad killer that face camps oh well. i die and move on to the next gam...
5,ENGLISH After playing for more than two years I am given the task of reviewing this game again.This review will be as complete as possible so if you do not want to see the full review I recommend ...,english after playing for more than two years i am given the task of reviewing this game again.this review will be as complete as possible so if you do not want to see the full review i recommend ...
6,Out of all the reviews I wrote This one is probably the most serious one I wrote. For starters the community of this game sucks just like every online game You don't wanna talk to them because the...,out of all the reviews i wrote this one is probably the most serious one i wrote. for starters the community of this game sucks just like every online game you don't wanna talk to them because the...
7,I have never been told to kill myself more than while playing this game.,i have never been told to kill myself more than while playing this game.
8,Any longtime Dead by Daylight player knows that this isn't a horror game. If you're looking for scares I wouldn't consider this to be a scary game because it becomes quite predictable over time. I...,any longtime dead by daylight player knows that this isn't a horror game. if you're looking for scares i wouldn't consider this to be a scary game because it becomes quite predictable over time. i...
9,if you think cs go is toxic try this game,if you think cs go is toxic try this game


## 5. Language Detection

### 5.1 Detect language

In [12]:
from lingua import Language, LanguageDetectorBuilder

languages = [Language.ENGLISH, Language.GERMAN, Language.FRENCH, Language.SPANISH,
             Language.ITALIAN, Language.PORTUGUESE, Language.RUSSIAN, Language.POLISH]

detector = LanguageDetectorBuilder.from_languages(*languages).build()

# Batch detection over whole column

results = detector.detect_languages_in_parallel_of(data['review_clean'].tolist())
data['language'] = [r.iso_code_639_1.name if r is not None else 'UNKNOWN' for r in results]

print(data['language'].value_counts(normalize = True).round(3))

language
EN         0.921
DE         0.018
UNKNOWN    0.017
PL         0.009
IT         0.009
PT         0.009
ES         0.008
FR         0.006
RU         0.003
Name: proportion, dtype: float64


### 5.2 Filter to english

In [13]:
before_len = len(data)

data = data[data['language'].isin(['EN', 'UNKNOWN'])].reset_index(drop = True)

print(f'Dropped {before_len - len(data)} non-English reviews')
print(f'Remaining: {len(data)}')

Dropped 26594 non-English reviews
Remaining: 406781


## 6. Length & Duplicate Flags

### 6.1 Word count

In [14]:
data['word_count'] = data['review_clean'].str.split().str.len()

In [15]:
data['is_duplicate'] = data.duplicated(subset = ['review_clean'], keep = 'first')
data.head()

,date_posted,funny,helpful,hour_played,is_early_access_review,recommendation,review,title,label,review_clean,language,word_count,is_duplicate
0,2019-02-10,2.0,4,578,False,Recommended,&gt Played as German Reich&gt Declare war on Belgium&gt Can't break Belgium so go through France&gt Capitulate France in order to get to Belgium&gt Get True Blitzkrieg achievementThis game is dad,Expansion - Hearts of Iron IV: Man the Guns,1,> played as german reich> declare war on belgium> can't break belgium so go through france> capitulate france in order to get to belgium> get true blitzkrieg achievementthis game is dad,EN,31,False
1,2019-02-10,0.0,0,184,False,Recommended,yes.,Expansion - Hearts of Iron IV: Man the Guns,1,yes.,EN,1,False
2,2019-02-07,0.0,0,892,False,Recommended,Very good game although a bit overpriced in my opinion. I'd prefer playing the game with mods (historical accuracy so on) although the vanilla version is good aswell. 7/10,Expansion - Hearts of Iron IV: Man the Guns,1,very good game although a bit overpriced in my opinion. i'd prefer playing the game with mods (historical accuracy so on) although the vanilla version is good aswell. 7/10,EN,29,False
3,2018-06-14,126.0,1086,676,False,Recommended,Out of all the reviews I wrote This one is probably the most serious one I wrote. For starters the community of this game sucks just like every online game You don't wanna talk to them because the...,Dead by Daylight,1,out of all the reviews i wrote this one is probably the most serious one i wrote. for starters the community of this game sucks just like every online game you don't wanna talk to them because the...,EN,419,False
4,2017-06-20,85.0,2139,612,False,Recommended,Disclaimer I survivor main. I play games for fun not for competition so the DBD community doesn't really get to me. If I get a bad killer that face camps oh well. I die and move on to the next gam...,Dead by Daylight,1,disclaimer i survivor main. i play games for fun not for competition so the dbd community doesn't really get to me. if i get a bad killer that face camps oh well. i die and move on to the next gam...,EN,273,False


In [16]:
print(data['is_duplicate'].value_counts())
print()
print(data.groupby('label')['word_count'].median())

is_duplicate
False    356718
True      50063
Name: count, dtype: int64

label
0    23.0
1    12.0
Name: word_count, dtype: float64


In [17]:
data['review_clean'].value_counts().head(20)

review_clean
good game         2659
good              1916
great game        1465
nice game         1329
nice              1013
.                  884
best game ever     870
10/10              773
yes                597
great game!        476
best game          467
awesome            398
very good game     392
fun game           380
)                  350
awesome game       331
very good          325
love it            322
< 3                320
amazing            320
Name: count, dtype: int64

## 7. Sanity Checks

### 7.1 Inspect cleaning results

In [18]:
# Before/after on a few rows
for raw, clean in data[['review', 'review_clean']].sample(5, random_state=42).values:
    print('RAW: ', repr(raw[:100]))
    print('CLEAN: ', repr(clean[:100]))
    print()

# Negattions must survive cleaning (critical for the sentiment variant)
for neg in ["not", "don't", "no", "never"]:
    n = data['review_clean'].str.contains(rf'\b{neg}\b', regex = True).sum()
    print(f'{neg!r}: {n} reviews')

RAW:  'Grand Theft Auto....More like Grand Theft My LifeAHAHAHAHAHAAHAHA...ahahaha....eeeehh... I am gunna '
CLEAN:  'grand theft auto....more like grand theft my lifeahahahahahaahaha...ahahaha....eeeehh... i am gunna '

RAW:  'Tried out Monster Hunter in hopes of it being something similar to Dragons Dogma as after watching a'
CLEAN:  'tried out monster hunter in hopes of it being something similar to dragons dogma as after watching a'

RAW:  'Bans team killing streamers who make the game popular but not the aimbotters who one shot you throug'
CLEAN:  'bans team killing streamers who make the game popular but not the aimbotters who one shot you throug'

RAW:  'I joined the GuLaG (Gunlance Gang).'
CLEAN:  'i joined the gulag (gunlance gang).'

RAW:  'everytime i switch to trev his eating out of a trash can.'
CLEAN:  'everytime i switch to trev his eating out of a trash can.'

'not': 63849 reviews
"don't": 27573 reviews
'no': 36215 reviews
'never': 15342 reviews


### Final Overview

In [19]:
print(f'Final shape: {data.shape}')
print()
data.info()
print()
print(data['label'].value_counts(normalize = True).round(3))

Final shape: (406781, 13)

<class 'pandas.DataFrame'>
RangeIndex: 406781 entries, 0 to 406780
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   date_posted             406781 non-null  str    
 1   funny                   406731 non-null  float64
 2   helpful                 406781 non-null  int64  
 3   hour_played             406781 non-null  int64  
 4   is_early_access_review  406781 non-null  bool   
 5   recommendation          406781 non-null  str    
 6   review                  406781 non-null  str    
 7   title                   406781 non-null  str    
 8   label                   406781 non-null  int64  
 9   review_clean            406781 non-null  str    
 10  language                406781 non-null  str    
 11  word_count              406781 non-null  int64  
 12  is_duplicate            406781 non-null  bool   
dtypes: bool(2), float64(1), int64(4), str(6)
memory usage: 231

## 8. Save clean data

In [23]:
data_clean_raw_data = data.drop(['date_posted', 'recommendation'], axis = 1)

data_clean_raw_data.to_parquet('../data/processed_data/reviews_clean_v2.parquet')

In [27]:
print(f'Saved {len(data_clean_raw_data)} rows, {data_clean_raw_data.shape[1]} columns')

Saved 406781 rows, 11 columns


In [25]:
cols_to_drop = ['date_posted', 'recommendation', 'review']

data_clean = data.drop(columns = cols_to_drop)
data_clean.to_parquet(
    '../data/processed_data/reviews_clean.parquet', index = False
)

print(f'Saved {len(data_clean)} rows, {data_clean.shape[1]} columns')

Saved 406781 rows, 10 columns


In [26]:
check = pd.read_parquet('../data/processed_data/reviews_clean.parquet')
print(check.shape)
print(check.dtypes)

(406781, 10)
funny                     float64
helpful                     int64
hour_played                 int64
is_early_access_review       bool
title                         str
label                       int64
review_clean                  str
language                      str
word_count                  int64
is_duplicate                 bool
dtype: object
